# NBA API — EDA v2: depth, restrictions & alternatives (Polars)

`initial_eda.ipynb` (v1) answered *"what does one `LeagueDashPlayerStats` season look like?"* This
follow-up answers the three questions that actually gate Statlas's data design — **empirically, against
the live API** (the docs are quirky and version-dependent, so we probe rather than trust):

1. **How far back does each endpoint go?** — season/advanced stats, tracking, shot detail, play-by-play.
2. **Are advanced metrics native or must we compute them?** — plus/minus, ratings, USG/TS/eFG, four factors, PIE, PER/BPM/VORP/WS.
3. **What other sources exist, and should we adopt any?** — Basketball-Reference, balldontlie, and others.

Every section ties back to the Statlas non-negotiables in `CLAUDE.md`: the engine's target schema is
`game_logs(player_id, season, game_date, opponent_abbr, is_playoff, min, pts, reb, ast, plus_minus)`,
**DuckDB does all arithmetic** (this notebook is for *understanding the source*, not computing product
numbers), and any leaderboard needs a **minimum-games qualifier**. Conventions match v1: `nba_api`
returns pandas, so we convert once with `pl.from_pandas(...)` and do everything in **Polars**; run via
`uv run`.

## §0 — Setup & reusable probe helpers

Two helpers carry the whole notebook. `safe_rows` wraps every live call in try/except + a polite sleep
(stats.nba.com rate-limits bursts), returning a row count so a flaky call never aborts the run.
`earliest_season` **binary-searches** the earliest season a season-list endpoint returns data for —
~7 calls instead of scanning 70+ years.

In [1]:
import os
import time

import nba_api
import polars as pl
from nba_api.stats.endpoints import (
    leaguedashplayerstats,
    leaguedashptstats,
    leaguegamefinder,
    leagueleaders,
    shotchartdetail,
)

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
print("nba_api", nba_api.__version__, "| polars", pl.__version__)

PAUSE = 0.6  # seconds between live calls — be polite to stats.nba.com


def season_str(end_year: int) -> str:
    """2025 -> '2024-25'. NBA labels a season by the calendar year it ENDS."""
    return f"{end_year - 1}-{str(end_year)[-2:]}"


def safe_rows(endpoint, **kwargs):
    """Call an nba_api endpoint; return (n_rows, polars_df_or_None, status).
    n_rows == -1 signals an error (vs 0 = a clean-but-empty response)."""
    try:
        resp = endpoint(timeout=30, **kwargs)
        pdf = resp.get_data_frames()[0]
        time.sleep(PAUSE)
        return len(pdf), pl.from_pandas(pdf), "ok"
    except Exception as e:  # noqa: BLE001 — the whole point is to never abort the run
        time.sleep(PAUSE)
        return -1, None, f"{type(e).__name__}: {str(e)[:70]}"


def earliest_season(probe, lo_year=1947, hi_year=2025):
    """Binary-search the earliest END-year where `probe(season)` returns > 0 rows.
    Assumes monotonicity: empty below the boundary, populated at/above it."""
    if probe(season_str(hi_year)) <= 0:
        return None  # nothing even at the top of the range
    lo, hi = lo_year, hi_year
    while lo < hi:
        mid = (lo + hi) // 2
        if probe(season_str(mid)) > 0:
            hi = mid
        else:
            lo = mid + 1
    return season_str(lo)

nba_api 1.11.4 | polars 1.41.2


## §1 — Q1: How far back does each endpoint go?

**Why it matters:** this is the literal ceiling on what Statlas can answer. A question about 1985 is
answerable from season totals but *not* from advanced ratings or shot charts. We binary-search each
**season-list** endpoint live (below), then bracket the two **game/player-scoped** endpoints
(shot charts, play-by-play) since those aren't keyed by a season list.

In [2]:
# Each probe returns ONLY a row count (what earliest_season needs).
def probe_ldps_base(s):
    return safe_rows(leaguedashplayerstats.LeagueDashPlayerStats,
                     season=s, measure_type_detailed_defense="Base")[0]

def probe_ldps_adv(s):
    return safe_rows(leaguedashplayerstats.LeagueDashPlayerStats,
                     season=s, measure_type_detailed_defense="Advanced")[0]

def probe_leaders(s):
    return safe_rows(leagueleaders.LeagueLeaders,
                     season=s, stat_category_abbreviation="PTS")[0]

def probe_tracking(s):
    return safe_rows(leaguedashptstats.LeagueDashPtStats,
                     season=s, player_or_team="Player", pt_measure_type="SpeedDistance")[0]

# lo_year just narrows the search window (fewer calls); 2025 is the known-populated top.
season_endpoints = [
    ("LeagueDashPlayerStats (Base)",     probe_ldps_base, 1990),
    ("LeagueDashPlayerStats (Advanced)", probe_ldps_adv,  1990),
    ("LeagueLeaders (basic totals)",     probe_leaders,   1947),
    ("LeagueDashPtStats (tracking)",     probe_tracking,  2005),
]

boundaries = {}
for name, probe, lo in season_endpoints:
    boundaries[name] = earliest_season(probe, lo_year=lo, hi_year=2025)
    print(f"{name:36s} earliest season with data = {boundaries[name]}")

LeagueDashPlayerStats (Base)         earliest season with data = 1996-97


LeagueDashPlayerStats (Advanced)     earliest season with data = 1996-97


LeagueLeaders (basic totals)         earliest season with data = 1951-52


LeagueDashPtStats (tracking)         earliest season with data = 2013-14


**Shot charts & play-by-play** are game/player-scoped, so we *bracket* the boundary (populated season
vs. the year before) instead of binary-searching — a full search would re-pull ~100k-row payloads each
step. `ShotChartDetail(player_id=0)` returns every shot league-wide for a season; `PlayByPlayV3` needs
a game id, so we pull one via `LeagueGameFinder` per era.

In [3]:
from nba_api.stats.endpoints import playbyplayv3

# --- ShotChartDetail: league-wide (player_id=0) at the suspected 1996-97 boundary ---
for s in ["1996-97", "1995-96"]:
    n, _, st = safe_rows(shotchartdetail.ShotChartDetail, team_id=0, player_id=0,
                         season_nullable=s, season_type_all_star="Regular Season",
                         context_measure_simple="FGA")
    print(f"ShotChartDetail {s}: shots = {n if n >= 0 else st}")

# --- PlayByPlayV3: grab the first game id of an era, then pull its PBP ---
def pbp_rows_for_season(season):
    n, games, st = safe_rows(leaguegamefinder.LeagueGameFinder,
                             season_nullable=season, league_id_nullable="00")
    if games is None or games.height == 0:
        return f"no games ({st})"
    gid = games["GAME_ID"][0]
    pn, _, pst = safe_rows(playbyplayv3.PlayByPlayV3, game_id=gid, start_period=0, end_period=14)
    return f"game {gid} -> {pn} events" if pn >= 0 else pst

for s in ["1996-97", "2023-24"]:
    print(f"PlayByPlayV3 {s}: {pbp_rows_for_season(s)}")

print("\nNOTE: PlayByPlayV2 and BoxScoreAdvancedV2 are DEPRECATED — they now return empty/KeyError.")
print("      Statlas must use the V3 variants (verified working above & in §2).")

ShotChartDetail 1996-97: shots = 106496


ShotChartDetail 1995-96: shots = 0


PlayByPlayV3 1996-97: game 0049600088 -> 482 events


PlayByPlayV3 2023-24: game 0042300405 -> 455 events

NOTE: PlayByPlayV2 and BoxScoreAdvancedV2 are DEPRECATED — they now return empty/KeyError.
      Statlas must use the V3 variants (verified working above & in §2).


### Which *stats* survive as you go back?

"Data exists" ≠ "every column exists." `LeagueLeaders` keeps a fixed schema across all eras, but the
tracked stats were introduced at different times — steals/blocks/turnovers in **1973-74**, the 3-point
line in **1979-80**. We detect this empirically: a stat is "live" for a season only if its column sums
to something non-zero. This is the *column-survival map* — it tells Statlas which questions are even
well-posed for a given year.

In [4]:
milestones = ["2024-25", "1996-97", "1979-80", "1973-74", "1970-71", "1951-52"]
tracked = ["STL", "BLK", "TOV", "FG3M", "OREB"]

rows = []
for s in milestones:
    n, df, st = safe_rows(leagueleaders.LeagueLeaders, season=s, stat_category_abbreviation="PTS")
    if df is None:
        rows.append({"season": s, "n_players": None, **{c: st for c in tracked}})
        continue
    rec = {"season": s, "n_players": df.height}
    for c in tracked:
        if c not in df.columns:
            rec[c] = "n/a"
        else:
            total = df.select(pl.col(c).cast(pl.Float64, strict=False).fill_null(0).sum()).item()
            rec[c] = "yes" if total and total > 0 else "— (zero/null)"
    rows.append(rec)

pl.DataFrame(rows)

season,n_players,STL,BLK,TOV,FG3M,OREB
str,i64,str,str,str,str,str
"""2024-25""",569,"""yes""","""yes""","""yes""","""yes""","""yes"""
"""1996-97""",433,"""yes""","""yes""","""yes""","""yes""","""yes"""
"""1979-80""",257,"""yes""","""yes""","""yes""","""yes""","""yes"""
"""1973-74""",211,"""yes""","""yes""","""— (zero/null)""","""— (zero/null)""","""yes"""
"""1970-71""",203,"""— (zero/null)""","""— (zero/null)""","""— (zero/null)""","""— (zero/null)""","""— (zero/null)"""
"""1951-52""",115,"""— (zero/null)""","""— (zero/null)""","""— (zero/null)""","""— (zero/null)""","""— (zero/null)"""


**→ Answer to Q1 (history is tiered):**

| Tier | Endpoints | Earliest | Statlas can ask about… |
|---|---|---|---|
| Tracking | `LeagueDashPtStats`, hustle/defense | **2013-14** | speed, distance, contested shots |
| Modern (cols rich) | `LeagueDashPlayerStats` Base+Advanced, `ShotChartDetail`, `PlayByPlayV3` | **1996-97** | ratings, USG/TS, shot locations, PBP |
| Basic totals | `LeagueLeaders` | **1951-52** | points/rebounds/assists (STL/BLK from 1973-74, 3PT from 1979-80) |

So Statlas can answer rich, advanced-stat questions for **1996-97→present**, basic counting-stat
questions back to **1951-52**, and tracking questions only for **2013-14→present**. The further back,
the fewer stats are well-posed — the resolver should *refuse* (not guess) a steals question about 1965.

## §2 — Q2: Are advanced metrics native, or must we compute them?

**Why it matters:** if a metric is native, Statlas just *ingests a column* and DuckDB serves it. If it's
derived, we'd have to compute it ourselves (more surface area, more ways to be wrong). We confirm what's
native by pulling the Advanced season table and one per-game `BoxScoreAdvancedV3`.

In [5]:
n, adv, st = safe_rows(leaguedashplayerstats.LeagueDashPlayerStats,
                       season="2024-25", measure_type_detailed_defense="Advanced")
native_adv = [c for c in adv.columns if c in {
    "OFF_RATING", "DEF_RATING", "NET_RATING", "AST_PCT", "OREB_PCT", "DREB_PCT",
    "REB_PCT", "USG_PCT", "TS_PCT", "EFG_PCT", "PACE", "PIE", "POSS",
}]
print("Native advanced columns present:", native_adv)
adv.select(["PLAYER_NAME", "GP", "OFF_RATING", "DEF_RATING", "NET_RATING",
            "USG_PCT", "TS_PCT", "PIE"]).sort("PIE", descending=True).head(8)

Native advanced columns present: ['OFF_RATING', 'DEF_RATING', 'NET_RATING', 'AST_PCT', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'EFG_PCT', 'TS_PCT', 'USG_PCT', 'PACE', 'PIE', 'POSS']


PLAYER_NAME,GP,OFF_RATING,DEF_RATING,NET_RATING,USG_PCT,TS_PCT,PIE
str,i64,f64,f64,f64,f64,f64,f64
"""Alondes Williams""",1,171.4,85.7,85.7,0.286,1.25,0.4
"""Giannis Antetokounmpo""",67,118.9,111.7,7.2,0.346,0.625,0.21
"""Nikola Jokić""",70,125.6,115.1,10.5,0.285,0.663,0.206
"""Leonard Miller""",13,79.2,98.6,-19.4,0.239,0.54,0.202
"""Shai Gilgeous-Alexander""",76,122.4,105.7,16.7,0.336,0.637,0.199
"""Anthony Davis""",51,113.3,112.9,0.5,0.301,0.588,0.179
"""Luka Dončić""",50,119.6,110.5,9.1,0.328,0.587,0.17
"""Joel Embiid""",19,104.0,109.5,-5.5,0.342,0.58,0.169


In [6]:
from nba_api.stats.endpoints import boxscoreadvancedv3

# A single recent game -> per-player advanced box (the V3 endpoint; V2 is dead).
n, games, st = safe_rows(leaguegamefinder.LeagueGameFinder,
                         season_nullable="2023-24", league_id_nullable="00")
gid = games["GAME_ID"][0]
bn, box, bst = safe_rows(boxscoreadvancedv3.BoxScoreAdvancedV3, game_id=gid)
print(f"BoxScoreAdvancedV3 game {gid}: {bn} player rows, status={bst}")
show = [c for c in ["firstName", "familyName", "offensiveRating", "defensiveRating",
                    "netRating", "usagePercentage", "trueShootingPercentage", "pie"]
        if c in box.columns]
box.select(show).head(6)

BoxScoreAdvancedV3 game 0042300405: 30 player rows, status=ok


firstName,familyName,offensiveRating,defensiveRating,netRating,usagePercentage,trueShootingPercentage
str,str,f64,f64,f64,f64,f64
"""Derrick""","""Jones Jr.""",95.1,117.5,-22.4,0.191,0.563
"""P.J.""","""Washington""",79.1,119.7,-40.6,0.149,0.286
"""Daniel""","""Gafford""",63.6,105.0,-41.4,0.2,0.6
"""Kyrie""","""Irving""",106.6,122.7,-16.1,0.212,0.444
"""Luka""","""Dončić""",98.8,120.5,-21.7,0.363,0.515
"""Dereck""","""Lively II""",127.3,140.9,-13.6,0.061,0.5


In [7]:
# The native-vs-derived verdict for every metric Statlas might want.
metric_rows = [
    ("Plus/Minus",            "native",  "PLUS_MINUS in Base box score",                 "already a game_logs column"),
    ("Off/Def/Net Rating",    "native",  "Advanced MeasureType / BoxScoreAdvancedV3",    "per-100-possession"),
    ("Usage % (USG%)",        "native",  "Advanced MeasureType",                          ""),
    ("True Shooting (TS%)",   "native",  "Advanced MeasureType",                          ""),
    ("Effective FG (eFG%)",   "native",  "Advanced MeasureType",                          ""),
    ("AST% / REB%",           "native",  "Advanced MeasureType",                          ""),
    ("Pace / Possessions",    "native",  "Advanced MeasureType (PACE, POSS)",             ""),
    ("Four Factors",          "native",  "BoxScoreFourFactorsV3",                          "eFG/TOV/OREB/FT rate"),
    ("PIE",                   "native",  "Advanced MeasureType",                          "NBA's all-in-one metric"),
    ("PER",                   "derived", "NOT NBA-provided",                              "Basketball-Reference"),
    ("BPM (Box Plus/Minus)",  "derived", "NOT NBA-provided",                              "Basketball-Reference"),
    ("VORP",                  "derived", "NOT NBA-provided",                              "Basketball-Reference"),
    ("Win Shares",            "derived", "NOT NBA-provided",                              "Basketball-Reference"),
]
pl.DataFrame(
    metric_rows,
    schema=["metric", "native_vs_derived", "nba_api_source", "note"],
    orient="row",
)

metric,native_vs_derived,nba_api_source,note
str,str,str,str
"""Plus/Minus""","""native""","""PLUS_MINUS in Base box score""","""already a game_logs column"""
"""Off/Def/Net Rating""","""native""","""Advanced MeasureType / BoxScor…","""per-100-possession"""
"""Usage % (USG%)""","""native""","""Advanced MeasureType""",""""""
"""True Shooting (TS%)""","""native""","""Advanced MeasureType""",""""""
"""Effective FG (eFG%)""","""native""","""Advanced MeasureType""",""""""
…,…,…,…
"""PIE""","""native""","""Advanced MeasureType""","""NBA's all-in-one metric"""
"""PER""","""derived""","""NOT NBA-provided""","""Basketball-Reference"""
"""BPM (Box Plus/Minus)""","""derived""","""NOT NBA-provided""","""Basketball-Reference"""


**→ Answer to Q2:** Nearly everything Statlas needs is **native** — plus/minus, the three ratings,
USG/TS/eFG, AST%/REB%, pace, four factors, and **PIE** all come straight off the API (1996-97→present).
The only gaps are **PER, BPM, VORP, Win Shares**, which the NBA simply doesn't publish (they're
Basketball-Reference inventions); PIE is the NBA's substitute. Crucially, "native" means Statlas
*ingests* these as columns — it does **not** recompute them. Per the `CLAUDE.md` non-negotiable, **DuckDB
still performs every arithmetic operation** at query time (sums, averages, min-games filtering); the
model never computes or recalls a stat.

## §3 — Q3: What other sources exist, and should we adopt any?

`nba_api` is free, keyless, and native-advanced — it stays **primary**. The open question is what
*complements* it: the derived metrics from §2, deeper history, and possession/lineup data. We demo the
two worth wiring up live, then tabulate the rest.

### Basketball-Reference (`basketball_reference_web_scraper`)

The one source that fills §2's gap: it serves **PER, BPM, VORP, Win Shares** directly, plus history to
the 1940s. The catch — it's a **scraper**: no API contract, fragile to HTML changes, rate-limited
(~20 req/min), and ToS-sensitive. Fine for an occasional enrichment pull, not a hot path.

In [8]:
try:
    from basketball_reference_web_scraper import client as bbr

    adv25 = bbr.players_advanced_season_totals(season_end_year=2025)  # 2024-25 season
    keep = ["name", "games_played", "player_efficiency_rating", "true_shooting_percentage",
            "usage_percentage", "box_plus_minus", "value_over_replacement_player", "win_shares"]
    clean = [{k: r[k] for k in keep} for r in adv25]
    bdf = pl.DataFrame(clean).rename({
        "player_efficiency_rating": "PER",
        "box_plus_minus": "BPM",
        "value_over_replacement_player": "VORP",
        "win_shares": "WS",
    })
    print(f"Basketball-Reference 2024-25 advanced totals: {bdf.height} players — metrics nba_api lacks:")
    print(bdf.filter(pl.col("games_played") >= 40)  # min-games qualifier, per the reliability rule
             .sort("VORP", descending=True)
             .select(["name", "games_played", "PER", "BPM", "VORP", "WS"]).head(8))
except Exception as e:  # noqa: BLE001 — scraping is fragile by nature
    print(f"Basketball-Reference scrape failed ({type(e).__name__}: {e}) — expected for a scraper; "
          "this is exactly the fragility caveat. Skipping, run continues.")

Basketball-Reference 2024-25 advanced totals: 654 players — metrics nba_api lacks:
shape: (8, 6)
┌─────────────────────────┬──────────────┬──────┬──────┬──────┬──────┐
│ name                    ┆ games_played ┆ PER  ┆ BPM  ┆ VORP ┆ WS   │
│ ---                     ┆ ---          ┆ ---  ┆ ---  ┆ ---  ┆ ---  │
│ str                     ┆ i64          ┆ f64  ┆ f64  ┆ f64  ┆ f64  │
╞═════════════════════════╪══════════════╪══════╪══════╪══════╪══════╡
│ Nikola Jokić            ┆ 70           ┆ 32.0 ┆ 13.3 ┆ 9.8  ┆ 16.4 │
│ Shai Gilgeous-Alexander ┆ 76           ┆ 30.7 ┆ 11.5 ┆ 8.9  ┆ 16.7 │
│ Giannis Antetokounmpo   ┆ 67           ┆ 30.5 ┆ 9.5  ┆ 6.6  ┆ 11.5 │
│ Tyrese Haliburton       ┆ 73           ┆ 21.8 ┆ 5.8  ┆ 4.9  ┆ 10.4 │
│ Jayson Tatum            ┆ 72           ┆ 21.7 ┆ 5.2  ┆ 4.8  ┆ 9.5  │
│ Stephen Curry           ┆ 70           ┆ 21.5 ┆ 6.3  ┆ 4.8  ┆ 7.9  │
│ LeBron James            ┆ 70           ┆ 22.7 ┆ 5.6  ┆ 4.7  ┆ 7.7  │
│ Anthony Edwards         ┆ 79           ┆ 20.1 ┆ 4

### balldontlie (`balldontlie`)

A clean JSON API for games/players/box scores — pleasant for quick prototypes, but **thin on
advanced/tracking data** and now **requires a free API key**. The cell self-skips when
`BALLDONTLIE_API_KEY` is unset so `uv run` stays green.

In [9]:
key = os.environ.get("BALLDONTLIE_API_KEY")
if not key:
    print("balldontlie: skipped — set BALLDONTLIE_API_KEY to run (free key at https://balldontlie.io).")
    print("Verdict regardless: convenient JSON, but no native advanced/tracking metrics — prototyping only.")
else:
    from balldontlie import BalldontlieAPI

    api = BalldontlieAPI(api_key=key)
    resp = api.nba.players.list(per_page=5)
    sample = [{"id": p.id, "name": f"{p.first_name} {p.last_name}", "team": getattr(p.team, "abbreviation", None)}
              for p in resp.data]
    print(pl.DataFrame(sample))

balldontlie: skipped — set BALLDONTLIE_API_KEY to run (free key at https://balldontlie.io).
Verdict regardless: convenient JSON, but no native advanced/tracking metrics — prototyping only.


### The rest — documented, not wired up

| Source | What it adds | Restrictions | Verdict for Statlas |
|---|---|---|---|
| **nba_api** (primary) | Native advanced + tracking + shot/PBP, 1996-97→ | Unofficial; rate-limits bursts; V2 endpoints dead | **Adopt — primary source** |
| **Basketball-Reference** (scraper) | PER/BPM/VORP/WS; history to 1940s | Scraping: fragile, ~20 req/min, ToS-sensitive | **Adopt as occasional complement** for derived metrics + deep history |
| **balldontlie** | Clean JSON games/players/box | Free key required; no advanced/tracking | Prototyping convenience only |
| **pbpstats** (pbpstats.com) | Possession-level, lineup & on/off | Smaller community project | Situational — if we add lineup data |
| **shufinskiy/nba_data** | Bulk PBP + shot detail (stats.nba.com, 1996-97→) | Static dumps; you manage refresh | Good for bulk backfill of PBP/shots |
| **hoopR** (sportsdataverse) | ESPN + NBA incl. PBP | **R-first**; Python access is second-class | Skip — wrong language for Statlas |
| **Kaggle NBA datasets** | Static historical CSVs | Frozen; no rate limits but stale | Handy for offline experiments |

**→ Answer to Q3:** Keep **nba_api primary**. Adopt **Basketball-Reference** as a *complement* — it's the
only practical way to get PER/BPM/VORP/WS and pre-1951 history — but treat it as an offline enrichment
job, never a request-path dependency (scraping fragility). Everything else is situational: balldontlie
for quick prototypes, shufinskiy/pbpstats if/when we want bulk PBP or lineup data, Kaggle for frozen
offline sets. hoopR is R-first — skip.

## §4 — Synthesis: does this populate the `game_logs` schema?

Everything above only matters if it fills Statlas's target table. `game_logs` is **game-level**, so the
season-table endpoints from §1–§2 inform *availability*, but the actual rows come from game-level
endpoints (`PlayerGameLog`, `LeagueGameFinder`, `BoxScore*V3`). Here's the column-by-column mapping.

In [10]:
game_logs_map = [
    ("player_id",     "native", "PlayerGameLog / BoxScore*V3 (PLAYER_ID)"),
    ("season",        "native", "query parameter / SEASON_ID"),
    ("game_date",     "native", "PlayerGameLog GAME_DATE"),
    ("opponent_abbr", "native", "PlayerGameLog MATCHUP (parse opponent tricode)"),
    ("is_playoff",    "native", "season_type_all_star='Playoffs' flag on the call"),
    ("min",           "native", "PlayerGameLog MIN"),
    ("pts",           "native", "PlayerGameLog PTS"),
    ("reb",           "native", "PlayerGameLog REB"),
    ("ast",           "native", "PlayerGameLog AST"),
    ("plus_minus",    "native", "PlayerGameLog PLUS_MINUS"),
]
print("Every game_logs column is natively available from nba_api, 1996-97 -> present:\n")
pl.DataFrame(game_logs_map, schema=["game_logs_column", "availability", "nba_api_source"], orient="row")

Every game_logs column is natively available from nba_api, 1996-97 -> present:



game_logs_column,availability,nba_api_source
str,str,str
"""player_id""","""native""","""PlayerGameLog / BoxScore*V3 (P…"
"""season""","""native""","""query parameter / SEASON_ID"""
"""game_date""","""native""","""PlayerGameLog GAME_DATE"""
"""opponent_abbr""","""native""","""PlayerGameLog MATCHUP (parse o…"
"""is_playoff""","""native""","""season_type_all_star='Playoffs…"
"""min""","""native""","""PlayerGameLog MIN"""
"""pts""","""native""","""PlayerGameLog PTS"""
"""reb""","""native""","""PlayerGameLog REB"""
"""ast""","""native""","""PlayerGameLog AST"""


**→ Bottom line for Statlas:**

- **Every `game_logs` column is natively served by `nba_api`, with no computation**, for **1996-97→present**
  — which is exactly the era where advanced stats and shot/PBP detail also exist. nba_api alone fully
  populates the warehouse for the modern era.
- **Basketball-Reference** is the only complement worth adopting, and only for what nba_api can't give:
  PER/BPM/VORP/WS and pre-1996 depth — as an offline enrichment, never on the request path.
- The **min-games qualifier** (here: `games_played >= 40`) still governs every leaderboard, whatever the
  source — a 6-game sample must not win.
- Operational gotcha to bake into the ingest layer: **use the V3 box-score/play-by-play endpoints**;
  the V2 variants are deprecated and silently return empty.

None of this changes the architecture: the database stays the calculator, the model stays language-only,
and the resolver refuses (doesn't guess) anything outside these proven boundaries.